# token_tracker.py 工作原理详解

本 notebook 逐步拆解 `scripts/token_tracker.py` 的完整设计：
- 它在 BioJSON 流水线中处于什么位置？解决什么问题？
- **Python class 封装基础**（重点教学）
- 每个方法具体做了什么？
- 全局单例模式是什么？
- 动手实战：不需要 API key，完整模拟使用流程
- 真实报告解读 + 在项目中的实际使用方式
- 关键 Python 知识点总结

In [ ]:
# ✅ 自包含示例：直接运行此 cell，不依赖前面任何 cell
import sys, json
sys.path.insert(0, '../scripts')
from token_tracker import TokenTracker

# ── Mock 对象：模拟 OpenAI response ─────────────────────────────
class MockUsage:
    def __init__(self, prompt_tokens, completion_tokens):
        self.prompt_tokens     = prompt_tokens
        self.completion_tokens = completion_tokens
        self.total_tokens      = prompt_tokens + completion_tokens

class MockResponse:
    """模拟 client.chat.completions.create() 的返回值"""
    def __init__(self, prompt_tokens, completion_tokens):
        self.usage = MockUsage(prompt_tokens, completion_tokens)

# ── 创建 tracker，模拟真实流水线 ────────────────────────────────
tracker = TokenTracker(model="Vendor2/Claude-4.6-opus")

# 阶段1: extract（数字来自真实报告 token_usage_extract_20260314_150807.json）
tracker.add(MockResponse(38030, 6451),
            stage="extract",
            file="MinerU_markdown_mmc3_2031567019886178304.md")

# 阶段2: verify（数字来自真实报告 token_usage_verify_20260314_170657.json）
tracker.add(MockResponse(40689, 3288), stage="verify", file="OsMYB110", gene="OsMYB110")
tracker.add(MockResponse(40697, 3136), stage="verify", file="OsMYB55",  gene="OsMYB55")
tracker.add(MockResponse(41137, 3092), stage="verify", file="OsDREB2A", gene="OsDREB2A")

# ── 查看 tracker.calls 的内容 ───────────────────────────────────
print("=== tracker.calls ===")
print(json.dumps(tracker.calls, indent=2, ensure_ascii=False))

# ── 打印汇总表格 ────────────────────────────────────────────────
tracker.print_summary()

# ── 手动验算 verify 的 prompt_tokens ───────────────────────────
verify_calls = [c for c in tracker.calls if c["stage"] == "verify"]
total_prompt = sum(c["prompt_tokens"] for c in verify_calls)
print(f"
verify prompt_tokens 验算: 40689+40697+41137 = {total_prompt}")
print(f"len(verify_calls) = {len(verify_calls)}  ← 这就是 summary 里的 calls 字段")

## 1. 总览：TokenTracker 是什么？为什么需要它？

### 1.1 在流水线中的位置

BioJSON 流水线每一步都要调用 LLM API，每次调用都会消耗 token（约等于花钱）。
`TokenTracker` 就是一个**计费追踪器**，记录每次 API 调用花了多少 token。

```
阶段1: md_to_json.py                    阶段2: verify_response.py

MD 原文 --[API调用]--> 提取JSON          提取JSON --[API调用]--> 验证JSON
           |                                          |
           v                                          v
     tracker.add(response,               tracker.add(response,
                 stage="extract")                     stage="verify")
           |                                          |
           +------------------+---------------------------+
                              |
                              v
                   tracker.print_summary()   <-- 汇总所有 token 用量
                   tracker.save_report()     <-- 保存为 JSON 报告
```

### 1.2 解决的核心问题

| 问题 | TokenTracker 怎么解决 |
|------|----------------------|
| 不知道一次运行花了多少 token | 自动累计每次 API 调用的 token 数 |
| 不知道哪个阶段最费 token | 按 stage 分类统计（extract / verify） |
| 不知道哪个文件最费 token | 每条记录关联文件名 |
| 运行完就忘了 token 用量 | 保存为 JSON 报告，永久留存 |
| 数字太大不直观 | 自动转换为 kTokens（千 token）单位 |

## 2. Python Class 基础回顾

在看 TokenTracker 的代码之前，先理解 Python class 的核心概念。

### 2.1 为什么需要 class？

假设我们要追踪 token 用量，**不用 class** 的话可能会这样写：

In [1]:
# 不用 class 的写法：用全局变量 + 函数

calls_list = []
model_name = "unknown"

def add_call(prompt_tokens, completion_tokens):
    calls_list.append({
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "total_tokens": prompt_tokens + completion_tokens,
    })

def print_total():
    total = sum(c["total_tokens"] for c in calls_list)
    print(f"模型: {model_name}, 总 token: {total}")

model_name = "GPT-4"
add_call(100, 50)
add_call(200, 80)
print_total()

模型: GPT-4, 总 token: 430


这样写能用，但有几个问题：

| 问题 | 说明 |
|------|------|
| 全局变量污染 | `calls_list` 和 `model_name` 暴露在全局，容易被意外修改 |
| 无法创建多个追踪器 | 如果想同时追踪两个不同模型的用量呢？只有一个 `calls_list` |
| 数据和操作分离 | `calls_list` 是数据，`add_call` 是操作，逻辑上属于一体但代码上分散 |

**class 就是把「数据 + 操作数据的函数」打包在一起。**

In [2]:
# 用 class 的写法

class SimpleTracker:
    """一个极简版的 token 追踪器"""
    
    def __init__(self, model):
        self.model = model
        self.calls = []
    
    def add(self, prompt_tokens, completion_tokens):
        self.calls.append({
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "total_tokens": prompt_tokens + completion_tokens,
        })
    
    def print_total(self):
        total = sum(c["total_tokens"] for c in self.calls)
        print(f"模型: {self.model}, 总 token: {total}")

# 现在可以轻松创建多个独立的追踪器！
tracker_a = SimpleTracker(model="GPT-4")
tracker_b = SimpleTracker(model="Claude-4")

tracker_a.add(100, 50)
tracker_a.add(200, 80)
tracker_b.add(300, 120)

tracker_a.print_total()  # 只统计 tracker_a 自己的调用
tracker_b.print_total()  # 只统计 tracker_b 自己的调用

模型: GPT-4, 总 token: 430
模型: Claude-4, 总 token: 420


### 2.2 核心概念速查表

| 概念 | 说明 | 对应代码 |
|------|------|----------|
| **class** | 定义一个蓝图/模板 | `class SimpleTracker:` |
| **对象/实例** | 根据模板创建的具体东西 | `tracker_a = SimpleTracker("GPT-4")` |
| **`__init__`** | 构造函数，创建对象时自动执行 | `def __init__(self, model):` |
| **`self`** | 指代当前这个对象自己 | `self.model = model` |
| **实例属性** | 每个对象独立拥有的数据 | `self.calls = []` |
| **方法** | 属于 class 的函数，第一个参数必须是 self | `def add(self, ...):` |

### 2.3 `self` 到底是什么？

`self` 是 Python 中最容易困惑的概念之一。简单理解：

```python
# 当你写：
tracker_a.add(100, 50)

# Python 内部实际执行的是：
SimpleTracker.add(tracker_a, 100, 50)
#                 ^^^^^^^^^
#                 tracker_a 被自动传给了 self 参数
```

所以 `self.calls` 就是 `tracker_a.calls`，是这个具体对象的数据。

In [ ]:
# 验证 self 的机制

tracker_c = SimpleTracker("Test")

# 写法1：正常调用
tracker_c.add(100, 50)

# 写法2：显式传 self（不推荐，但能帮你理解）
SimpleTracker.add(tracker_c, 200, 80)

print(f"tracker_c 现在有 {len(tracker_c.calls)} 条记录")
tracker_c.print_total()

### 2.4 `__init__` 的作用

`__init__` 是构造函数（初始化函数），在 `SimpleTracker("GPT-4")` 时自动被调用。

它的职责是**给新对象设置初始状态**。

In [3]:
class Demo:
    def __init__(self, name):
        print(f"  __init__ 被调用了！name={name}")
        self.name = name
        self.data = []

print("创建 obj1：")
obj1 = Demo("Alice")

print("创建 obj2：")
obj2 = Demo("Bob")

print(f"\nobj1.name = {obj1.name}")
print(f"obj2.name = {obj2.name}")
print(f"obj1.data is obj2.data? {obj1.data is obj2.data}")  # False，各自独立

创建 obj1：
  __init__ 被调用了！name=Alice
创建 obj2：
  __init__ 被调用了！name=Bob

obj1.name = Alice
obj2.name = Bob
obj1.data is obj2.data? False


好了，有了这些 class 基础，现在来看真正的 `TokenTracker`！

## 3. 逐方法拆解 TokenTracker

| 方法 | 作用 | 类型 |
|------|------|------|
| `__init__(model)` | 初始化：设置模型名 + 空调用列表 | 构造函数 |
| `add(response, stage, file, gene)` | 从 API 响应中提取 token 用量并记录 | 核心方法 |
| `_aggregate(stage_filter)` | 按阶段聚合统计 token 数 | 内部方法 |
| `get_summary()` | 获取所有阶段的汇总数据 | 查询方法 |
| `print_summary()` | 格式化打印用量汇总表 | 输出方法 |
| `save_report(path)` | 保存完整报告为 JSON 文件 | 输出方法 |
| `merge(other)` | 合并另一个 tracker 的记录 | 工具方法 |

### 3.1 `__init__(self, model="unknown")` -- 初始化

```python
class TokenTracker:
    def __init__(self, model="unknown"):
        self.model = model      # 记录使用的模型名
        self.calls = []         # 存储每次 API 调用的详细记录
```

非常简洁，只设置了两个属性：
- `self.model`：字符串，模型名称（用于报告显示）
- `self.calls`：列表，每个元素是一次 API 调用的字典记录

**设计思考：** `model` 有默认值 `"unknown"`，所以 `TokenTracker()` 不传参数也能创建。

In [4]:
import sys
sys.path.insert(0, '../scripts')

from token_tracker import TokenTracker

tracker = TokenTracker(model="Vendor2/Claude-4.6-opus")
print(f"tracker.model = {tracker.model}")
print(f"tracker.calls = {tracker.calls}")
print(f"len(tracker.calls) = {len(tracker.calls)}")

# 不传 model 也行
tracker_default = TokenTracker()
print(f"\n默认 model = {tracker_default.model}")

tracker.model = Vendor2/Claude-4.6-opus
tracker.calls = []
len(tracker.calls) = 0

默认 model = unknown


### 3.2 `add(self, response, stage, file, gene)` -- 记录一次 API 调用

这是最常用的方法。每次调用 API 后，把 response 对象传给它，它自动提取 token 用量。

```python
def add(self, response, stage="unknown", file="", gene=""):
    usage = getattr(response, "usage", None)  # 安全取属性
    if usage is None:
        return  # 不崩溃，只是跳过
    record = {
        "stage": stage, "file": file,
        "prompt_tokens": usage.prompt_tokens or 0,
        "completion_tokens": usage.completion_tokens or 0,
        "total_tokens": usage.total_tokens or 0,
        "timestamp": datetime.now().isoformat(),
    }
    if gene:
        record["gene"] = gene
    self.calls.append(record)
```

#### 关键知识点：`getattr(obj, attr, default)`

安全地获取对象属性。如果属性不存在，返回 default 而不是报错。

In [5]:
# getattr 详解

class Fruit:
    def __init__(self):
        self.color = "red"

apple = Fruit()
print(f"apple.color = {apple.color}")
print(f"getattr(apple, 'color', None) = {getattr(apple, 'color', None)}")
print(f"getattr(apple, 'weight', None) = {getattr(apple, 'weight', None)}")

print("\n在 TokenTracker 中：如果 response 没有 usage 属性，getattr 返回 None，不崩溃。")

apple.color = red
getattr(apple, 'color', None) = red
getattr(apple, 'weight', None) = None

在 TokenTracker 中：如果 response 没有 usage 属性，getattr 返回 None，不崩溃。


In [6]:
# 模拟 API response 对象来演示 add()

class MockUsage:
    def __init__(self, prompt_tokens, completion_tokens, total_tokens):
        self.prompt_tokens = prompt_tokens
        self.completion_tokens = completion_tokens
        self.total_tokens = total_tokens

class MockResponse:
    def __init__(self, prompt_tokens, completion_tokens):
        total = prompt_tokens + completion_tokens
        self.usage = MockUsage(prompt_tokens, completion_tokens, total)

# 创建 tracker
tracker = TokenTracker(model="Vendor2/Claude-4.6-opus")

# 模拟 3 次 API 调用
r1 = MockResponse(1500, 800)
tracker.add(r1, stage="extract", file="paper_A.md")

r2 = MockResponse(2000, 1200)
tracker.add(r2, stage="extract", file="paper_B.md")

r3 = MockResponse(3000, 500)
tracker.add(r3, stage="verify", file="paper_A.md", gene="ZmPSY1")

print(f"共 {len(tracker.calls)} 条记录：\n")
for i, c in enumerate(tracker.calls):
    gene_info = f", gene={c['gene']}" if 'gene' in c else ''
    print(f"  [{i}] {c['stage']:>8} | {c['file']:<14} | {c['total_tokens']:>5} tokens{gene_info}")

共 3 条记录：

  [0]  extract | paper_A.md     |  2300 tokens
  [1]  extract | paper_B.md     |  3200 tokens
  [2]   verify | paper_A.md     |  3500 tokens, gene=ZmPSY1


In [ ]:
# 演示 usage 不可用的情况

class BadResponse:
    pass  # 没有 usage 属性

print("传入没有 usage 的 response：")
tracker.add(BadResponse(), stage="extract", file="broken.md")
print(f"记录数仍然是 {len(tracker.calls)}（没有新增，不会崩溃）")

### 3.3 `_aggregate(self, stage_filter=None)` -- 内部聚合方法

按阶段汇总 token 用量，返回一个字典。

**为什么方法名以 `_` 开头？** 这是 Python 命名约定，表示「内部方法」。外部应该用 `get_summary()` 而不是直接调 `_aggregate()`。Python 不会强制禁止，但这是好习惯。

#### 列表推导式

`[c for c in self.calls if c["stage"] == "extract"]` 等价于：

```python
result = []
for c in self.calls:
    if c["stage"] == "extract":
        result.append(c)
```

In [7]:
# 列表推导式详解

calls = [
    {"stage": "extract", "total_tokens": 2300},
    {"stage": "extract", "total_tokens": 3200},
    {"stage": "verify",  "total_tokens": 3500},
]

# 列表推导式写法
extract_calls = [c for c in calls if c["stage"] == "extract"]
print(f"extract 阶段: {len(extract_calls)} 条")

# sum + 生成器表达式
total = sum(c["total_tokens"] for c in extract_calls)
print(f"extract 总 token: {total}")

extract 阶段: 2 条
extract 总 token: 5500


In [ ]:
# 演示 _aggregate
print("=== 聚合 extract 阶段 ===")
for k, v in tracker._aggregate("extract").items():
    print(f"  {k}: {v}")

print("\n=== 聚合全部 ===")
for k, v in tracker._aggregate().items():
    print(f"  {k}: {v}")

### 3.4 `get_summary()` -- 获取汇总数据

```python
def get_summary(self):
    stages = sorted(set(c["stage"] for c in self.calls))
    summary = {}
    for stage in stages:
        summary[stage] = self._aggregate(stage)
    summary["total"] = self._aggregate()
    return summary
```

用 `set()` 去重自动发现所有 stage，然后逐个聚合。

In [ ]:
import json
summary = tracker.get_summary()
print(json.dumps(summary, indent=2, ensure_ascii=False))

### 3.5 `print_summary()` -- 格式化打印汇总表

#### f-string 对齐语法速查

| 语法 | 含义 |
|------|------|
| `{x:<12}` | 左对齐，占 12 字符宽 |
| `{x:>10}` | 右对齐，占 10 字符宽 |
| `{x:>10.2f}` | 右对齐，宽 10，保留 2 位小数 |

In [ ]:
# f-string 对齐演示
name = "extract"
value = 3.5

print(f"左对齐: [{name:<12}]")
print(f"右对齐: [{value:>10.2f}]")

In [ ]:
# 调用 print_summary 看效果
tracker.print_summary()

### 3.6 `save_report(path)` -- 保存 JSON 报告

把 model、calls 列表、summary 汇总打包成 JSON 保存。

关键点：`os.makedirs(..., exist_ok=True)` 自动创建目录，已存在也不报错。

In [ ]:
import tempfile, os

tmp_path = os.path.join(tempfile.gettempdir(), "demo_token_report.json")
tracker.save_report(tmp_path)

# 查看保存的内容
with open(tmp_path) as f:
    saved = json.load(f)

print("保存的 JSON 结构：")
print(f"  顶层 key: {list(saved.keys())}")
print(f"  model: {saved["model"]}")
print(f"  calls 数量: {len(saved["calls"])}")
print(f"  summary stages: {list(saved["summary"].keys())}")

### 3.7 `merge(other)` -- 合并两个 tracker

```python
def merge(self, other):
    if isinstance(other, TokenTracker):
        self.calls.extend(other.calls)
```

`isinstance(other, TokenTracker)` 检查 other 是不是 TokenTracker 类型，防止传入错误对象。

使用场景：提取和验证分别用不同 tracker，最后合并统一报告。

In [ ]:
# merge 演示
tracker_extract = TokenTracker(model="Claude")
tracker_extract.add(MockResponse(1000, 500), stage='extract', file='a.md')
tracker_extract.add(MockResponse(1200, 600), stage='extract', file='b.md')

tracker_verify = TokenTracker(model="Claude")
tracker_verify.add(MockResponse(2000, 300), stage='verify', file='a.md')

print(f"合并前: extract={len(tracker_extract.calls)}, verify={len(tracker_verify.calls)}")

tracker_extract.merge(tracker_verify)
print(f"合并后: extract tracker 共 {len(tracker_extract.calls)} 条")
tracker_extract.print_summary()

## 4. 全局单例模式

文件末尾还有三个**模块级函数**（不属于 class）：

```python
_global_tracker = None

def get_tracker(model="unknown"):
    global _global_tracker
    if _global_tracker is None:
        _global_tracker = TokenTracker(model=model)
    return _global_tracker

def reset_tracker():
    global _global_tracker
    _global_tracker = None
```

### 什么是单例模式？

**单例（Singleton）** 指的是整个程序中只有一个实例。

为什么需要？假设 `md_to_json.py` 中有多个函数都需要记录 token：

```python
# 不用单例：需要把 tracker 当参数到处传递
def process_file(md_path, tracker):
    ...
    tracker.add(response, ...)

def main():
    tracker = TokenTracker(...)
    process_file('a.md', tracker)  # 必须传进去
    process_file('b.md', tracker)  # 必须传进去
```

```python
# 用单例：任何地方直接 get_tracker() 拿到同一个对象
from token_tracker import get_tracker

def process_file(md_path):
    tracker = get_tracker()  # 拿到全局唯一的那个
    ...
    tracker.add(response, ...)
```

### `global` 关键字

在函数内部要**修改**模块级变量时，必须用 `global` 声明。否则 Python 会把它当成局部变量。

In [ ]:
from token_tracker import get_tracker, reset_tracker

# 重置确保干净
reset_tracker()

# 第一次调用：创建新的 tracker
t1 = get_tracker(model="TestModel")
print(f"t1.model = {t1.model}")
print(f"t1 id = {id(t1)}")

# 第二次调用：返回同一个对象（不会创建新的）
t2 = get_tracker()
print(f"\nt2 id = {id(t2)}")
print(f"t1 is t2? {t1 is t2}")  # True！同一个对象

# 重置后再获取：会创建新的
reset_tracker()
t3 = get_tracker(model="NewModel")
print(f"\n重置后: t3.model = {t3.model}")
print(f"t1 is t3? {t1 is t3}")  # False，不同对象了

## 5. 动手实战：模拟完整使用流程

下面模拟一个完整的 BioJSON 运行过程（不需要 API key）。

In [8]:
# 完整模拟
tracker = TokenTracker(model="Vendor2/Claude-4.6-opus")

# === 阶段1：提取 ===
print("=== 阶段1：提取 ===")
files = [
    ("paper_cell_2017.md", 8500, 2200),
    ("paper_nature_2022.md", 12000, 3500),
    ("paper_tieman2017.md", 6800, 1800),
]
for fname, p, c in files:
    resp = MockResponse(p, c)
    tracker.add(resp, stage="extract", file=fname)
    print(f"  {fname}: {p+c} tokens")

# === 阶段2：验证（每个文件多个基因）===
print("\n=== 阶段2：验证 ===")
verifications = [
    ("paper_cell_2017.md", "ZmPSY1", 4000, 800),
    ("paper_cell_2017.md", "ZmCRTISO", 4200, 900),
    ("paper_nature_2022.md", "SlMYB12", 5000, 700),
]
for fname, gene, p, c in verifications:
    resp = MockResponse(p, c)
    tracker.add(resp, stage="verify", file=fname, gene=gene)
    print(f"  {fname} / {gene}: {p+c} tokens")

# === 汇总 ===
tracker.print_summary()

=== 阶段1：提取 ===
  paper_cell_2017.md: 10700 tokens
  paper_nature_2022.md: 15500 tokens
  paper_tieman2017.md: 8600 tokens

=== 阶段2：验证 ===
  paper_cell_2017.md / ZmPSY1: 4800 tokens
  paper_cell_2017.md / ZmCRTISO: 5100 tokens
  paper_nature_2022.md / SlMYB12: 5700 tokens

════════════════════════════════════════════════════════════
📊 API Token 用量汇总 (模型: Vendor2/Claude-4.6-opus)
════════════════════════════════════════════════════════════
  阶段              输入 (kT)    输出 (kT)    合计 (kT)     调用次数
  ──────────── ────────── ────────── ────────── ────────
  extract           27.30       7.50      34.80        3
  verify            13.20       2.40      15.60        3
  ──────────── ────────── ────────── ────────── ────────
  总计                40.50       9.90      50.40        6
════════════════════════════════════════════════════════════


## 6. 真实报告解读

加载项目中的真实 token 用量报告。

In [ ]:
# 加载真实报告
report_path = "../reports/token_usage_demo.json"

try:
    with open(report_path, "r") as f:
        report = json.load(f)
    print(f"模型: {report['model']}")
    print(f"时间: {report['timestamp']}")
    print(f"调用次数: {len(report['calls'])}")
    print()
    for call in report['calls']:
        print(f"  stage={call['stage']}, file={call['file']}, tokens={call['total_tokens']}")
    print()
    print("汇总：")
    for stage, data in report['summary'].items():
        print(f"  {stage}: {data['total_ktokens']} kTokens ({data['calls']} calls)")
except FileNotFoundError:
    print(f"报告文件不存在: {report_path}")

## 7. 在项目中的实际使用

### md_to_json.py 中

```python
from token_tracker import TokenTracker

tracker = TokenTracker(model=os.getenv("MODEL", "Vendor2/Claude-4.6-opus"))

# 每次 API 调用后
response = client.chat.completions.create(...)
tracker.add(response, stage="extract", file=md_file)

# 脚本结束时
tracker.print_summary()
tracker.save_report("reports/token_usage_extract_xxx.json")
```

### verify_response.py 中

```python
from token_tracker import TokenTracker

tracker = TokenTracker(model=MODEL)

# 每验证一个基因
response = client.chat.completions.create(...)
tracker.add(response, stage="verify", file=stem, gene=gene_name)

# 验证结束
tracker.print_summary()
tracker.save_report("reports/token_usage_verify_xxx.json")
```

## 8. 关键 Python 知识点总结

| 知识点 | 在 TokenTracker 中的使用 | 说明 |
|--------|-------------------------|------|
| `class` | `class TokenTracker:` | 把数据和操作封装在一起 |
| `__init__` | 初始化 model 和 calls | 创建对象时自动调用 |
| `self` | 所有方法的第一个参数 | 指代当前对象实例 |
| `getattr()` | 安全获取 response.usage | 属性不存在时返回默认值 |
| 列表推导式 | 筛选特定 stage 的调用 | `[c for c in ... if ...]` |
| `_` 前缀 | `_aggregate()` 标记为内部方法 | Python 命名约定 |
| `isinstance()` | merge 时检查类型 | 防止传入错误类型 |
| `global` | 全局单例的实现 | 函数内修改模块级变量 |
| `os.makedirs(exist_ok=True)` | save_report 创建目录 | 目录已存在也不报错 |
| f-string 对齐 | print_summary 表格格式化 | `{x:<12}` `{x:>10.2f}` |

### 类比总结

如果把 TokenTracker 比作一个**记账本**：

| 类比 | TokenTracker |
|------|-------------|
| 记账本封面写的名字 | `self.model` |
| 记账本里的每一行 | `self.calls` 列表中的每个字典 |
| 记一笔账 | `add()` 方法 |
| 按类别算小计 | `_aggregate(stage)` |
| 看总览页 | `get_summary()` / `print_summary()` |
| 拍照存档 | `save_report()` |
| 把两本账合并 | `merge()` |
| 公司唯一的一本账 | 全局单例 `get_tracker()` |